# Lesson 10 - Agents IA en production

Dans cette leçon, vous apprendrez les **modèles de production** pour les agents IA en utilisant le Microsoft Agent Framework (Python). Nous couvrons :

- **Observabilité** — ajout du suivi temporel et des logs aux interactions de l'agent
- **Évaluation** — utilisation d'un agent évaluateur pour noter la qualité des réponses
- **Gestion des coûts** — stratégies d'optimisation des tokens et de sélection des modèles

Le scénario est un **agent de voyage** qui aide les utilisateurs à planifier des voyages, avec une supervision et une évaluation ajoutées par-dessus.


## Installation


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## Considérations de production

Passer des agents IA des prototypes à la production nécessite une attention particulière à trois piliers :

1. **Observabilité** — Vous devez avoir une visibilité sur ce que fait l’agent, combien de temps cela prend et quels outils il utilise. Sans traçage ni journalisation, le débogage des problèmes en production est presque impossible.

2. **Évaluation** — Des contrôles qualité automatisés garantissent que les réponses de l’agent restent précises, complètes et utiles dans le temps. Un agent évaluateur peut noter les réponses selon des critères définis.

3. **Gestion des coûts** — L’utilisation des tokens impacte directement les coûts. Des stratégies comme l’optimisation des prompts, le choix des modèles et la mise en cache aident à maîtriser les dépenses sans sacrifier la qualité.


## Construire un agent observable

Nous définissons des outils de voyage et enveloppons l'appel d'agent avec un chronométrage afin de pouvoir surveiller la latence. En production, vous intégreriez OpenTelemetry ou un système de traçage similaire.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Modèles d'évaluation

Un modèle courant en production est d'utiliser un second agent comme **évaluateur**. L'évaluateur note la réponse de l'agent principal selon des critères prédéfinis tels que la complétude, la précision et l'utilité.

Cela permet :
- Des barrières de qualité automatisées avant que les réponses n'atteignent les utilisateurs
- La détection des régressions lors des changements de prompts ou de modèles
- La surveillance continue des performances de l'agent au fil du temps


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Stratégies de gestion des coûts

Le contrôle des coûts est essentiel pour les agents d'IA de production. Voici les principales stratégies :

| Strategy | Description |
|---|---|
| **Prompt optimization** | Gardez les instructions système concises. Supprimez le contexte redondant pour réduire les jetons d'entrée. |
| **Model selection** | Utilisez des modèles plus petits et moins coûteux (par ex. GPT-4o-mini) pour des tâches simples comme la classification ou l'extraction, et réservez les modèles plus grands pour un raisonnement complexe. |
| **Caching** | Mettez en cache les résultats des outils et les requêtes fréquentes pour éviter les appels API redondants. |
| **Token budgets** | Définissez des limites `max_tokens` pour éviter des réponses trop longues inattendues. |
| **Batching** | Regroupez plusieurs requêtes utilisateur en un seul appel API lorsque cela est possible. |

En pratique, une approche graduée fonctionne bien : dirigez les demandes simples vers un modèle rapide et peu coûteux et n'escaladez les requêtes complexes qu'à un modèle plus performant.


## Résumé

Dans cette leçon, vous avez appris à :

1. **Ajouter de l’observabilité** aux interactions des agents avec le chronométrage et la journalisation, posant ainsi les bases du traçage et de la surveillance.
2. **Évaluer automatiquement les réponses des agents** à l’aide d’un agent évaluateur qui note la complétude, la précision et l’utilité.
3. **Gérer les coûts** grâce à l’optimisation des instructions, le choix du modèle, le cache et les budgets en tokens.

Ces modèles de production aident à garantir que vos agents d’IA sont fiables, mesurables et rentables à grande échelle.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Avertissement** :  
Ce document a été traduit à l’aide du service de traduction automatisée [Co-op Translator](https://github.com/Azure/co-op-translator). Bien que nous nous efforcions d’assurer l’exactitude, veuillez noter que les traductions automatiques peuvent contenir des erreurs ou des inexactitudes. Le document original dans sa langue d’origine doit être considéré comme la source faisant foi. Pour des informations critiques, une traduction professionnelle humaine est recommandée. Nous déclinons toute responsabilité en cas de malentendus ou d’interprétations erronées résultant de l’utilisation de cette traduction.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
